In [ ]:
# NIPAH VIRUS RISK STRATIFICATION SYSTEM
# Complete Five-Phase Implementation for Google Colab / Jupyter Notebook
# Author: Muhammad Owais Raza Qadri
# Date: 21 March 2026

# INSTALLATION (Run once)
"""
!pip install -U numpy pandas scikit-learn imbalanced-learn xgboost lightgbm shap matplotlib seaborn ipywidgets

"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, classification_report,
                             precision_recall_curve, average_precision_score, brier_score_loss)
from sklearn.calibration import calibration_curve

from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from imblearn.pipeline import make_pipeline as make_imbalance_pipeline

import xgboost as xgb
import lightgbm as lgb
import shap
import ipywidgets as widgets
from IPython.display import display, clear_output

# Set random seed for reproducibility

np.random.seed(42)

print("=" * 80)
print("NIPAH VIRUS RISK STRATIFICATION SYSTEM")
print("Five-Phase Machine Learning Implementation")
print("Research and Code By Muhammad Owais Raza Qadri")
print("=" * 80)

# GENERATE SYNTHETIC DATASET (Based on 2020-2026 Epidemiology)

def generate_nipah_dataset(n_samples=2216, pos_ratio=0.325):
    """
    Generate synthetic Nipah virus dataset based on 2020-2026 epidemiological profiles
    """
    n_pos = int(n_samples * pos_ratio)
    n_neg = n_samples - n_pos

    data = []

    # Generate positive cases (Nipah confirmed)
    for i in range(n_pos):
        # Clinical features
        fever = 1
        fever_duration = np.random.randint(2, 8)
        headache = np.random.choice([0, 1], p=[0.2, 0.8])
        cough = np.random.choice([0, 1], p=[0.3, 0.7])
        dyspnea = np.random.choice([0, 1], p=[0.3, 0.7])
        respiratory_rate = np.random.randint(18, 35) if dyspnea else np.random.randint(12, 20)
        o2_saturation = np.random.randint(85, 98) if dyspnea else np.random.randint(95, 100)

        # Neurological symptoms (key differentiator)
        altered_sensorium = np.random.choice([0, 1], p=[0.2, 0.8])
        gcs = np.random.randint(8, 15) if altered_sensorium else 15
        seizures = np.random.choice([0, 1], p=[0.6, 0.4])
        focal_neuro = np.random.choice([0, 1], p=[0.7, 0.3])

        # Epidemiological exposures
        contact_case = np.random.choice([0, 1], p=[0.4, 0.6])
        healthcare_worker = np.random.choice([0, 1], p=[0.7, 0.3])
        bat_exposure = np.random.choice([0, 1], p=[0.4, 0.6])
        date_palm_sap = np.random.choice([0, 1], p=[0.5, 0.5])
        endemic_region = np.random.choice([0, 1], p=[0.2, 0.8])

        # Laboratory
        thrombocytopenia = np.random.choice([0, 1], p=[0.3, 0.7])
        platelet_count = np.random.randint(20000, 120000) if thrombocytopenia else np.random.randint(150000, 400000)

        # Demographics
        age = np.random.randint(5, 75)
        gender = np.random.choice([0, 1])

        # Composite scores
        neuro_score = (altered_sensorium * 2 + seizures * 3 + focal_neuro * 2 + (gcs < 12) * 3)
        resp_score = (cough * 1 + dyspnea * 2 + (respiratory_rate > 24) * 2 + (o2_saturation < 94) * 3)
        exposure_score = (contact_case * 3 + healthcare_worker * 2 + bat_exposure * 2 + date_palm_sap * 3)

        # Outcome
        nipah_positive = 1

        patient = {
            'age': age, 'gender': gender,
            'fever_duration': fever_duration,
            'cough': cough, 'dyspnea': dyspnea,
            'respiratory_rate': respiratory_rate, 'o2_saturation': o2_saturation,
            'altered_sensorium': altered_sensorium, 'gcs': gcs,
            'seizures': seizures, 'focal_neuro': focal_neuro,
            'contact_case': contact_case, 'healthcare_worker': healthcare_worker,
            'bat_exposure': bat_exposure, 'date_palm_sap': date_palm_sap,
            'endemic_region': endemic_region,
            'thrombocytopenia': thrombocytopenia, 'platelet_count': platelet_count,
            'neuro_score': neuro_score, 'resp_score': resp_score, 'exposure_score': exposure_score,
            'nipah_positive': nipah_positive
        }
        data.append(patient)

    # Generate negative cases (other febrile illnesses)
    for i in range(n_neg):
        # Clinical features - less neurological involvement
        fever = 1
        fever_duration = np.random.randint(1, 5)
        headache = np.random.choice([0, 1], p=[0.3, 0.7])
        cough = np.random.choice([0, 1], p=[0.5, 0.5])
        dyspnea = np.random.choice([0, 1], p=[0.8, 0.2])
        respiratory_rate = np.random.randint(14, 25) if dyspnea else np.random.randint(12, 18)
        o2_saturation = np.random.randint(92, 100) if dyspnea else np.random.randint(96, 100)

        # Neurological - much less common
        altered_sensorium = np.random.choice([0, 1], p=[0.9, 0.1])
        gcs = np.random.randint(13, 15) if altered_sensorium else 15
        seizures = np.random.choice([0, 1], p=[0.98, 0.02])
        focal_neuro = np.random.choice([0, 1], p=[0.99, 0.01])

        # Epidemiological exposures - lower probability
        contact_case = np.random.choice([0, 1], p=[0.9, 0.1])
        healthcare_worker = np.random.choice([0, 1], p=[0.9, 0.1])
        bat_exposure = np.random.choice([0, 1], p=[0.7, 0.3])
        date_palm_sap = np.random.choice([0, 1], p=[0.8, 0.2])
        endemic_region = np.random.choice([0, 1], p=[0.3, 0.7])

        # Laboratory
        thrombocytopenia = np.random.choice([0, 1], p=[0.8, 0.2])
        platelet_count = np.random.randint(80000, 200000) if thrombocytopenia else np.random.randint(150000, 450000)

        # Demographics
        age = np.random.randint(1, 85)
        gender = np.random.choice([0, 1])

        # Composite scores
        neuro_score = (altered_sensorium * 2 + seizures * 3 + focal_neuro * 2 + (gcs < 12) * 3)
        resp_score = (cough * 1 + dyspnea * 2 + (respiratory_rate > 24) * 2 + (o2_saturation < 94) * 3)
        exposure_score = (contact_case * 3 + healthcare_worker * 2 + bat_exposure * 2 + date_palm_sap * 3)

        # Outcome
        nipah_positive = 0

        patient = {
            'age': age, 'gender': gender,
            'fever_duration': fever_duration,
            'cough': cough, 'dyspnea': dyspnea,
            'respiratory_rate': respiratory_rate, 'o2_saturation': o2_saturation,
            'altered_sensorium': altered_sensorium, 'gcs': gcs,
            'seizures': seizures, 'focal_neuro': focal_neuro,
            'contact_case': contact_case, 'healthcare_worker': healthcare_worker,
            'bat_exposure': bat_exposure, 'date_palm_sap': date_palm_sap,
            'endemic_region': endemic_region,
            'thrombocytopenia': thrombocytopenia, 'platelet_count': platelet_count,
            'neuro_score': neuro_score, 'resp_score': resp_score, 'exposure_score': exposure_score,
            'nipah_positive': nipah_positive
        }
        data.append(patient)

    df = pd.DataFrame(data)
    return df

# Generate dataset
print("\n[PHASE 2] Generating synthetic Nipah virus dataset...")
df = generate_nipah_dataset(n_samples=2216, pos_ratio=0.325)
print(f"Dataset generated: {df.shape[0]} patients, {df.shape[1]} features")
print(f"Positive cases (Nipah confirmed): {df['nipah_positive'].sum()} ({df['nipah_positive'].mean()*100:.1f}%)")
print(f"Negative cases: {(1-df['nipah_positive']).sum()} ({(1-df['nipah_positive']).mean()*100:.1f}%)")


# EXPLORATORY DATA ANALYSIS

print("\n[PHASE 2] Exploratory Data Analysis")

# Correlation with target
correlations = df.corr()['nipah_positive'].sort_values(ascending=False)
print("\nTop features correlated with Nipah positive status:")
print(correlations[1:11])

# PREPARE FEATURES AND TARGET

feature_columns = ['age', 'gender', 'fever_duration', 'cough', 'dyspnea',
                   'respiratory_rate', 'o2_saturation', 'altered_sensorium',
                   'gcs', 'seizures', 'focal_neuro', 'contact_case',
                   'healthcare_worker', 'bat_exposure', 'date_palm_sap',
                   'endemic_region', 'thrombocytopenia', 'platelet_count',
                   'neuro_score', 'resp_score', 'exposure_score']

X = df[feature_columns]
y = df['nipah_positive']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set: {X_train.shape[0]} patients")
print(f"Test set: {X_test.shape[0]} patients")

# CLASS IMBALANCE HANDLING WITH SMOTE-TOMEK

print("\n[PHASE 2] Applying SMOTE-TOMEK for class imbalance...")

smote_tomek = SMOTETomek(random_state=42)
X_train_resampled, y_train_resampled = smote_tomek.fit_resample(X_train, y_train)

print(f"Before resampling - Training set: {X_train.shape[0]}, Positive: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"After SMOTE-TOMEK - Training set: {X_train_resampled.shape[0]}, Positive: {y_train_resampled.sum()} ({y_train_resampled.mean()*100:.1f}%)")

# FEATURE SCALING

numerical_cols = ['age', 'fever_duration', 'respiratory_rate', 'o2_saturation',
                  'gcs', 'platelet_count', 'neuro_score', 'resp_score', 'exposure_score']
binary_cols = [col for col in feature_columns if col not in numerical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('binary', 'passthrough', binary_cols)
    ])

# PHASE 3: MODEL DEVELOPMENT

print("\n" + "=" * 80)
print("PHASE 3: MODEL DEVELOPMENT")
print("=" * 80)

# Define models
models = {
    'Logistic Regression': LogisticRegression(C=0.1, penalty='l1', solver='saga',
                                               class_weight='balanced', max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=15,
                                            min_samples_split=5, class_weight='balanced', random_state=42),
    'XGBoost': xgb.XGBClassifier(learning_rate=0.05, max_depth=6, subsample=0.8,
                                 n_estimators=400, random_state=42, eval_metric='logloss'),
    'LightGBM': lgb.LGBMClassifier(num_leaves=31, learning_rate=0.03,
                                   n_estimators=500, random_state=42, verbose=-1),
    'SVM': SVC(C=10, gamma=0.05, kernel='rbf', probability=True,
               class_weight='balanced', random_state=42)
}

# Train and evaluate each model
results = []

for name, model in models.items():
    print(f"\nTraining {name}...")

    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

    pipeline.fit(X_train_resampled, y_train_resampled)

    y_pred = pipeline.predict(X_test)
    y_pred_proba = pipeline.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    sensitivity = recall_score(y_test, y_pred)
    specificity = recall_score(y_test, y_pred, pos_label=0)
    precision = precision_score(y_test, y_pred)
    auc_roc = roc_auc_score(y_test, y_pred_proba)

    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Sensitivity': sensitivity,
        'Specificity': specificity,
        'Precision': precision,
        'AUC-ROC': auc_roc
    })

    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Sensitivity: {sensitivity:.4f}")
    print(f"  Specificity: {specificity:.4f}")
    print(f"  AUC-ROC: {auc_roc:.4f}")

# Display results
results_df = pd.DataFrame(results)
print("\n" + "-" * 60)
print("PHASE 3 RESULTS: MODEL COMPARISON")
print("-" * 60)
print(results_df.to_string(index=False))

# PHASE 4: MODEL EVALUATION

print("\n" + "=" * 80)
print("PHASE 4: MODEL EVALUATION")
print("=" * 80)

# Select best model (LightGBM)
best_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', lgb.LGBMClassifier(num_leaves=31, learning_rate=0.03,
                                      n_estimators=500, random_state=42, verbose=-1))
])
best_model.fit(X_train_resampled, y_train_resampled)

y_pred_best = best_model.predict(X_test)
y_pred_proba_best = best_model.predict_proba(X_test)[:, 1]

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
print("\nConfusion Matrix:")
print(f"              Predicted Negative  Predicted Positive")
print(f"Actual Negative     {cm[0,0]:4d}              {cm[0,1]:4d}")
print(f"Actual Positive     {cm[1,0]:4d}              {cm[1,1]:4d}")

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best, target_names=['Negative', 'Positive']))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba_best)
auc = roc_auc_score(y_test, y_pred_proba_best)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'LightGBM (AUC = {auc:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('ROC Curve - LightGBM Model')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('roc_curve.png', dpi=100)
plt.show()
print("[Figure: ROC Curve saved as 'roc_curve.png']")

# MODEL INTERPRETATION WITH SHAP

print("\n[PHASE 4] Generating SHAP explanations...")

# Get preprocessed feature names
preprocessor.fit(X_train_resampled)
feature_names = numerical_cols + binary_cols

# Get LightGBM model
lgb_model = best_model.named_steps['classifier']

# Create SHAP explainer
explainer = shap.TreeExplainer(lgb_model)

# Transform test data
X_test_transformed = preprocessor.transform(X_test)
X_test_df = pd.DataFrame(X_test_transformed, columns=feature_names)

# Calculate SHAP values (sample for speed)
X_test_sample = X_test_df.head(100)
shap_values = explainer.shap_values(X_test_sample)

# Summary plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_sample, show=False)
plt.title('SHAP Feature Importance Summary')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=100)
plt.show()
print("[Figure: SHAP summary saved as 'shap_summary.png']")

# PHASE 5: CLINICAL IMPLEMENTATION - INTERACTIVE PREDICTOR

print("\n" + "=" * 80)
print("PHASE 5: CLINICAL IMPLEMENTATION - INTERACTIVE PREDICTOR")
print("=" * 80)

def predict_nipah_risk(patient_name, father_name, age, fever_days, cough, dyspnea,
                       altered_sensorium, seizures, contact_case, bat_exposure,
                       date_palm_sap, endemic_region, thrombocytopenia):
    """
    Predict Nipah virus risk for a new patient
    """
    # Create input dataframe
    input_df = pd.DataFrame({
        'age': [age],
        'gender': [0],
        'fever_duration': [fever_days],
        'cough': [1 if cough else 0],
        'dyspnea': [1 if dyspnea else 0],
        'respiratory_rate': [22],
        'o2_saturation': [95],
        'altered_sensorium': [1 if altered_sensorium else 0],
        'gcs': [13 if altered_sensorium else 15],
        'seizures': [1 if seizures else 0],
        'focal_neuro': [0],
        'contact_case': [1 if contact_case else 0],
        'healthcare_worker': [0],
        'bat_exposure': [1 if bat_exposure else 0],
        'date_palm_sap': [1 if date_palm_sap else 0],
        'endemic_region': [1 if endemic_region else 0],
        'thrombocytopenia': [1 if thrombocytopenia else 0],
        'platelet_count': [85000 if thrombocytopenia else 250000],
        'neuro_score': [(1 if altered_sensorium else 0)*2 + (1 if seizures else 0)*3],
        'resp_score': [(1 if cough else 0)*1 + (1 if dyspnea else 0)*2],
        'exposure_score': [(1 if contact_case else 0)*3 + (1 if bat_exposure else 0)*2 + (1 if date_palm_sap else 0)*3]
    })

    # Predict probability
    probability = best_model.predict_proba(input_df)[0, 1]

    # Assign risk category
    if probability >= 0.7:
        risk_category = "HIGH"
        color = "RED"
        recommendation = ("URGENT: Order RT-PCR testing immediately. "
                         "Implement droplet and contact precautions. "
                         "Isolate patient in designated area. "
                         "Consult infectious disease specialist.")
    elif probability >= 0.3:
        risk_category = "INTERMEDIATE"
        color = "YELLOW"
        recommendation = ("Monitor closely for symptom progression. "
                         "Schedule confirmatory testing within 24-48 hours. "
                         "Educate patient/family on warning signs. "
                         "Enhanced vigilance required.")
    else:
        risk_category = "LOW"
        color = "GREEN"
        recommendation = ("Standard clinical management for presenting symptoms. "
                         "Routine follow-up. "
                         "No specific Nipah testing indicated at this time.")

    # Display result
    clear_output(wait=True)
    print("\n" + "=" * 60)
    print("NIPAH VIRUS RISK ASSESSMENT RESULT")
    print("=" * 60)
    print(f"Patient Name: {patient_name}")
    print(f"Father's Name: {father_name}")
    print(f"Age: {age} years")
    print("-" * 60)
    print(f"Predicted Probability: {probability:.1%}")
    print(f"Risk Category: {risk_category} [{color}]")
    print("-" * 60)
    print("RECOMMENDED ACTIONS:")
    print(recommendation)
    print("=" * 60)

    # Key drivers (simplified explanation)
    print("\nKey Risk Factors Identified:")
    if altered_sensorium:
        print("• Neurological symptoms (altered sensorium)")
    if seizures:
        print("• Seizures (high-risk neurological manifestation)")
    if contact_case:
        print("• Known contact with confirmed case")
    if date_palm_sap:
        print("• Date palm sap consumption history")
    if bat_exposure:
        print("• Bat habitat exposure")
    if endemic_region:
        print("• Located in endemic region")
    if thrombocytopenia:
        print("• Thrombocytopenia (low platelets)")

    return probability, risk_category

# Create interactive widgets
print("\n[PHASE 5] Launching Interactive Risk Predictor...")
print("Please fill in the patient details below:")

name_widget = widgets.Text(description="Patient Name:", value="Ahmed Khan")
father_widget = widgets.Text(description="Father's Name:", value="Abdullah Khan")
age_widget = widgets.IntSlider(description="Age:", min=1, max=90, value=35)
fever_widget = widgets.IntSlider(description="Fever Days:", min=1, max=14, value=5)
cough_widget = widgets.Checkbox(description="Cough", value=True)
dyspnea_widget = widgets.Checkbox(description="Difficulty Breathing", value=False)
altered_sensorium_widget = widgets.Checkbox(description="Altered Sensorium", value=True)
seizures_widget = widgets.Checkbox(description="Seizures", value=False)
contact_widget = widgets.Checkbox(description="Contact with Confirmed Case", value=True)
bat_widget = widgets.Checkbox(description="Bat Exposure", value=True)
sap_widget = widgets.Checkbox(description="Date Palm Sap Consumption", value=False)
endemic_widget = widgets.Checkbox(description="Endemic Region (Kerala/WB)", value=True)
thrombocytopenia_widget = widgets.Checkbox(description="Thrombocytopenia (Low Platelets)", value=True)

predict_button = widgets.Button(description="PREDICT RISK", button_style='danger')

# Define button click handler
def on_predict_clicked(b):
    predict_nipah_risk(
        name_widget.value,
        father_widget.value,
        age_widget.value,
        fever_widget.value,
        cough_widget.value,
        dyspnea_widget.value,
        altered_sensorium_widget.value,
        seizures_widget.value,
        contact_widget.value,
        bat_widget.value,
        sap_widget.value,
        endemic_widget.value,
        thrombocytopenia_widget.value
    )

predict_button.on_click(on_predict_clicked)

# Display widgets
display(name_widget, father_widget, age_widget, fever_widget,
        cough_widget, dyspnea_widget, altered_sensorium_widget,
        seizures_widget, contact_widget, bat_widget, sap_widget,
        endemic_widget, thrombocytopenia_widget, predict_button)

print("\n" + "=" * 80)
print("FIVE-PHASE IMPLEMENTATION COMPLETE")
print("=" * 80)
print("\nInstructions:")
print("1. Adjust patient symptoms using the checkboxes above")
print("2. Click 'PREDICT RISK' to see risk assessment")
print("3. The system provides probability, risk category, and clinical recommendations")
print("4. Key risk factors are displayed for clinical interpretation")


NIPAH VIRUS RISK ASSESSMENT RESULT
Patient Name: Hamza
Father's Name: Majeedi
Age: 60 years
------------------------------------------------------------
Predicted Probability: 100.0%
Risk Category: HIGH [RED]
------------------------------------------------------------
RECOMMENDED ACTIONS:
URGENT: Order RT-PCR testing immediately. Implement droplet and contact precautions. Isolate patient in designated area. Consult infectious disease specialist.

Key Risk Factors Identified:
• Neurological symptoms (altered sensorium)
• Known contact with confirmed case
• Date palm sap consumption history
